In [1]:
# INSTALACIÓN DE PYSPARK
!pip install -q pyspark

In [2]:
# CREACIÓN DE LA SESIÓN SPARK
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Atenciones_SanMartin") \
    .getOrCreate()

print("Sesión Spark creada correctamente")
print("Versión:", spark.version)

Sesión Spark creada correctamente
Versión: 4.0.4


In [3]:
# CARGA DEL ARCHIVO CSV
from google.colab import files

archivos = files.upload()
ruta = next(iter(archivos))

# VERIFICACIÓN DE LAS PRIMERAS LÍNEAS
with open(ruta, encoding="utf-8-sig") as archivo:
    for _ in range(3):
        print(archivo.readline().strip())

Saving atenciones_spark.csv to atenciones_spark.csv
PROVINCIA;ATENDIDOS;ATENCIONES
MOYOBAMBA;0;2
MOYOBAMBA;0;2


In [4]:
# LECTURA DEL CSV COMO DATAFRAME
datos = spark.read \
    .option("header", True) \
    .option("sep", ";") \
    .schema("PROVINCIA STRING, ATENDIDOS LONG, ATENCIONES LONG") \
    .csv(ruta)

# VERIFICACIÓN DE LOS DATOS
print("Registros cargados:", datos.count())
datos.show(10, truncate=False)

Registros cargados: 1048410
+----------+---------+----------+
|PROVINCIA |ATENDIDOS|ATENCIONES|
+----------+---------+----------+
|MOYOBAMBA |0        |2         |
|MOYOBAMBA |0        |2         |
|MOYOBAMBA |0        |2         |
|MOYOBAMBA |0        |2         |
|BELLAVISTA|0        |1         |
|BELLAVISTA|0        |1         |
|BELLAVISTA|0        |1         |
|BELLAVISTA|0        |1         |
|BELLAVISTA|0        |1         |
|BELLAVISTA|0        |1         |
+----------+---------+----------+
only showing top 10 rows


In [5]:
# AGREGACIÓN DE ATENCIONES POR PROVINCIA
from pyspark.sql import functions as F

resultado_df = datos.groupBy("PROVINCIA").agg(
    F.sum("ATENCIONES").alias("TotalAtenciones")
).orderBy(F.desc("TotalAtenciones"), "PROVINCIA")

# VISUALIZACIÓN DEL RESULTADO
resultado_df.show(truncate=False)

+----------------+---------------+
|PROVINCIA       |TotalAtenciones|
+----------------+---------------+
|SAN MARTIN      |305251         |
|RIOJA           |278898         |
|MOYOBAMBA       |234044         |
|TOCACHE         |167305         |
|BELLAVISTA      |119783         |
|LAMAS           |116563         |
|MARISCAL CACERES|93596          |
|EL DORADO       |76037          |
|HUALLAGA        |73558          |
|PICOTA          |61239          |
+----------------+---------------+



In [6]:
# CREACIÓN DE LA VISTA TEMPORAL
datos.createOrReplaceTempView("atenciones")

# AGREGACIÓN CON SPARK SQL
resultado_sql = spark.sql("""
    SELECT PROVINCIA,
           SUM(ATENCIONES) AS TotalAtenciones
    FROM atenciones
    GROUP BY PROVINCIA
    ORDER BY TotalAtenciones DESC, PROVINCIA
""")

# VISUALIZACIÓN DEL RESULTADO
resultado_sql.show(truncate=False)

+----------------+---------------+
|PROVINCIA       |TotalAtenciones|
+----------------+---------------+
|SAN MARTIN      |305251         |
|RIOJA           |278898         |
|MOYOBAMBA       |234044         |
|TOCACHE         |167305         |
|BELLAVISTA      |119783         |
|LAMAS           |116563         |
|MARISCAL CACERES|93596          |
|EL DORADO       |76037          |
|HUALLAGA        |73558          |
|PICOTA          |61239          |
+----------------+---------------+



In [8]:
# MEDICIÓN DE TRES CONSULTAS NUEVAS POR ALTERNATIVA
from time import perf_counter

for nombre in ["DataFrame", "Spark SQL"]:
    tiempos = []

    for i in range(3):
        inicio = perf_counter()

        if nombre == "DataFrame":
            consulta = datos.groupBy("PROVINCIA").agg(
                F.sum("ATENCIONES").alias("TotalAtenciones")
            ).orderBy(F.desc("TotalAtenciones"), "PROVINCIA")
        else:
            consulta = spark.sql("""
                SELECT PROVINCIA,
                       SUM(ATENCIONES) AS TotalAtenciones
                FROM atenciones
                GROUP BY PROVINCIA
                ORDER BY TotalAtenciones DESC, PROVINCIA
            """)

        consulta.collect()
        segundos = perf_counter() - inicio
        tiempos.append(segundos)
        print(f"{nombre} — ejecución {i + 1}: {segundos:.3f} s")

    print(f"Promedio: {sum(tiempos) / 3:.3f} s\n")

DataFrame — ejecución 1: 1.971 s
DataFrame — ejecución 2: 2.331 s
DataFrame — ejecución 3: 1.815 s
Promedio: 2.039 s

Spark SQL — ejecución 1: 1.934 s
Spark SQL — ejecución 2: 2.375 s
Spark SQL — ejecución 3: 3.044 s
Promedio: 2.451 s

